# Benchmark Forecasting — Coûts GCP Veolia

Ce notebook prend en entrée les fichiers exportés par `Timeseries.ipynb` et compare **5 approches** de prévision de séries temporelles.

| # | Modèle | Famille | Librairie |
|---|--------|---------|----------|
| 1 | **AutoARIMA** | Statistique classique | Nixtla `statsforecast` |
| 2 | **AutoETS + AutoTheta** | Lissage exponentiel | Nixtla `statsforecast` |
| 3 | **TimesNet** | Deep Learning (2D variation) | Nixtla `neuralforecast` |
| 4 | **N-HiTS** | Neural hierarchical interpolation | Nixtla `neuralforecast` |
| 5 | **Prophet** | Additif décomposable | Meta (awesome-ts list) |

**Métriques** : MAE, RMSE, MAPE, R²

> **Prérequis** : lancer la cellule `Export pour Benchmark_Forecasting.ipynb` du notebook `Timeseries.ipynb` pour générer `daily_costs.parquet` et `daily_per_service.parquet`.

In [ ]:
# Installation des dépendances (à lancer une seule fois)
# !pip install statsforecast neuralforecast prophet scikit-learn plotly pandas pyarrow

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

print('Imports de base OK')

## 0. Chargement des données

In [ ]:
# Chargement depuis les parquets exportés par Timeseries.ipynb
daily = pd.read_parquet('daily_costs.parquet')
daily['ds'] = pd.to_datetime(daily['ds'])
daily = daily.sort_values('ds').reset_index(drop=True)

per_service = pd.read_parquet('daily_per_service.parquet')
per_service['ds'] = pd.to_datetime(per_service['ds'])
per_service = per_service.sort_values('ds').reset_index(drop=True)

print(f'  Série journalière : {len(daily)} jours  ({daily["ds"].min().date()} → {daily["ds"].max().date()})')
print(f'  Services disponibles : {len(per_service.columns) - 1}')
daily.head()

In [ ]:
# ── Split train / test ────────────────────────────────────────────────────────
HORIZON = 30          # jours à prédire
n_total  = len(daily)
n_test   = HORIZON
n_train  = n_total - n_test

train = daily.iloc[:n_train].copy()
test  = daily.iloc[n_train:].copy()

print(f'  Train : {n_train} jours  ({train["ds"].min().date()} → {train["ds"].max().date()})')
print(f'  Test  : {n_test} jours   ({test["ds"].min().date()}  → {test["ds"].max().date()})')

# Visualisation du split
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train',
                         line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines', name='Test (ground truth)',
                         line=dict(color='orange')))
fig.add_vrect(x0=test['ds'].min(), x1=test['ds'].max(),
              fillcolor='rgba(255,165,0,0.08)', line_width=0,
              annotation_text=f'Zone test ({n_test}j)', annotation_position='top left')
fig.update_layout(title='Split Train / Test — Coût journalier total',
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

In [ ]:
# ── Métriques ─────────────────────────────────────────────────────────────────
def compute_metrics(y_true, y_pred, model_name):
    y_true = np.array(y_true, dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / np.where(y_true == 0, 1e-9, y_true))) * 100
    r2   = r2_score(y_true, y_pred)
    return {'Model': model_name, 'MAE': round(mae, 2), 'RMSE': round(rmse, 2),
            'MAPE (%)': round(mape, 2), 'R²': round(r2, 4)}

results = []   # accumulateur des métriques
forecasts_all = {}   # stockage des prévisions pour la viz finale
y_test = test['y'].values
dates_test = test['ds'].values

print('  Métriques initialisées — prêt pour les modèles.')

---
## Modèle 1 — AutoARIMA  *(Nixtla StatsForecast)*

AutoARIMA sélectionne automatiquement les ordres `(p,d,q)(P,D,Q)s` par minimisation de l'AIC. Adapté aux séries avec tendance et/ou saisonnalité hebdomadaire (`season_length=7`).

In [ ]:
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA

# Format Nixtla : unique_id, ds, y
train_sf = train.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

sf_arima = StatsForecast(
    models=[AutoARIMA(season_length=7, approximation=True)],
    freq='D',
    n_jobs=-1,
    verbose=False
)
sf_arima.fit(train_sf)
fc_arima = sf_arima.predict(h=HORIZON)

y_pred_arima = fc_arima['AutoARIMA'].values
metrics_arima = compute_metrics(y_test, y_pred_arima, 'AutoARIMA')
results.append(metrics_arima)
forecasts_all['AutoARIMA'] = y_pred_arima

print(f"  AutoARIMA — MAE={metrics_arima['MAE']:.2f}€  RMSE={metrics_arima['RMSE']:.2f}€  "
      f"MAPE={metrics_arima['MAPE (%)']:.1f}%  R²={metrics_arima['R²']:.4f}")

# Graphique
fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_arima, mode='lines', name='AutoARIMA',
                         line=dict(color='crimson', dash='dash')))
fig.update_layout(title=f"AutoARIMA | R²={metrics_arima['R²']:.4f} | MAE={metrics_arima['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

---
## Modèle 2 — AutoETS + AutoTheta  *(Nixtla StatsForecast)*

- **AutoETS** : sélection automatique des composantes Erreur/Tendance/Saisonnalité.
- **AutoTheta** : variante de la méthode Theta, efficace sur séries courtes.

In [ ]:
from statsforecast.models import AutoETS, AutoTheta

sf_ets = StatsForecast(
    models=[AutoETS(season_length=7), AutoTheta(season_length=7)],
    freq='D',
    n_jobs=-1,
    verbose=False
)
sf_ets.fit(train_sf)
fc_ets = sf_ets.predict(h=HORIZON)

y_pred_ets   = fc_ets['AutoETS'].values
y_pred_theta = fc_ets['AutoTheta'].values

for name, pred in [('AutoETS', y_pred_ets), ('AutoTheta', y_pred_theta)]:
    m = compute_metrics(y_test, pred, name)
    results.append(m)
    forecasts_all[name] = pred
    print(f"  {name:<12} — MAE={m['MAE']:.2f}€  RMSE={m['RMSE']:.2f}€  MAPE={m['MAPE (%)']:.1f}%  R²={m['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_ets,   mode='lines', name='AutoETS',   line=dict(color='green',  dash='dash')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_theta, mode='lines', name='AutoTheta', line=dict(color='purple', dash='dot')))
fig.update_layout(title='AutoETS vs AutoTheta', xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

---
## Modèle 3 — TimesNet  *(Nixtla NeuralForecast)*

TimesNet transforme la série 1D en représentation **2D** pour capturer les variations intra-période et inter-période via des blocs `TimesBlock` (convolutions 2D). Architecture proposée par Wu et al. (2023) — *TimesNet: Temporal 2D-Variation Modeling for General Time Series Analysis*.

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import TimesNet

# Format NeuralForecast : unique_id, ds, y
train_nf = train.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

INPUT_SIZE = 2 * HORIZON  # fenêtre de contexte = 2× l'horizon

timesnet_model = TimesNet(
    h=HORIZON,
    input_size=INPUT_SIZE,
    encoder_layers=2,
    d_model=32,
    d_ff=64,
    dropout=0.1,
    top_k=3,
    num_kernels=4,
    max_steps=300,
    batch_size=8,
    learning_rate=1e-3,
    random_seed=42,
    scaler_type='standard',
)

nf_timesnet = NeuralForecast(models=[timesnet_model], freq='D')
nf_timesnet.fit(train_nf)
fc_timesnet = nf_timesnet.predict()

y_pred_timesnet = fc_timesnet['TimesNet'].values
metrics_timesnet = compute_metrics(y_test, y_pred_timesnet, 'TimesNet')
results.append(metrics_timesnet)
forecasts_all['TimesNet'] = y_pred_timesnet

print(f"  TimesNet — MAE={metrics_timesnet['MAE']:.2f}€  RMSE={metrics_timesnet['RMSE']:.2f}€  "
      f"MAPE={metrics_timesnet['MAPE (%)']:.1f}%  R²={metrics_timesnet['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_timesnet, mode='lines', name='TimesNet',
                         line=dict(color='darkred', dash='dash')))
fig.update_layout(title=f"TimesNet | R²={metrics_timesnet['R²']:.4f} | MAE={metrics_timesnet['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

---
## Modèle 4 — N-HiTS  *(Nixtla NeuralForecast)*

**N-HiTS** (Neural Hierarchical Interpolation for Time Series) décompose la prévision en plusieurs échelles temporelles grâce à du *hierarchical interpolation*, ce qui lui permet d'être très efficace sur des séries longues ou multi-saisonnières. Challu et al. (2023).

In [ ]:
from neuralforecast.models import NHITS

nhits_model = NHITS(
    h=HORIZON,
    input_size=2 * HORIZON,
    stack_types=['identity', 'identity', 'identity'],
    n_blocks=[1, 1, 1],
    mlp_units=[[256, 256], [256, 256], [256, 256]],
    interpolation_mode='linear',
    dropout_prob_theta=0.1,
    max_steps=300,
    batch_size=8,
    learning_rate=1e-3,
    random_seed=42,
    scaler_type='standard',
)

nf_nhits = NeuralForecast(models=[nhits_model], freq='D')
nf_nhits.fit(train_nf)
fc_nhits = nf_nhits.predict()

y_pred_nhits = fc_nhits['NHITS'].values
metrics_nhits = compute_metrics(y_test, y_pred_nhits, 'N-HiTS')
results.append(metrics_nhits)
forecasts_all['N-HiTS'] = y_pred_nhits

print(f"  N-HiTS — MAE={metrics_nhits['MAE']:.2f}€  RMSE={metrics_nhits['RMSE']:.2f}€  "
      f"MAPE={metrics_nhits['MAPE (%)']:.1f}%  R²={metrics_nhits['R²']:.4f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(x=test['ds'],  y=y_pred_nhits, mode='lines', name='N-HiTS',
                         line=dict(color='teal', dash='dash')))
fig.update_layout(title=f"N-HiTS | R²={metrics_nhits['R²']:.4f} | MAE={metrics_nhits['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

---
## Modèle 5 — Prophet  *(Meta / awesome-time-series-forecasting)*

**Prophet** modélise la série comme une somme de composantes : tendance + saisonnalité hebdomadaire + saisonnalité annuelle + jours fériés. Robuste aux valeurs manquantes et aux outliers. Figurant en tête des ressources du repo *awesome-time-series-forecasting*.

In [ ]:
from prophet import Prophet

train_prophet = train.rename(columns={'ds': 'ds', 'y': 'y'})

# Données insuffisantes pour saisonnalité annuelle (6 mois) → on désactive
prophet_model = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.1,
    seasonality_prior_scale=5,
)
prophet_model.fit(train_prophet)

future = prophet_model.make_future_dataframe(periods=HORIZON)
fc_prophet = prophet_model.predict(future)

# Extraire uniquement la zone test
y_pred_prophet = fc_prophet['yhat'].iloc[-HORIZON:].values

metrics_prophet = compute_metrics(y_test, y_pred_prophet, 'Prophet')
results.append(metrics_prophet)
forecasts_all['Prophet'] = y_pred_prophet

print(f"  Prophet — MAE={metrics_prophet['MAE']:.2f}€  RMSE={metrics_prophet['RMSE']:.2f}€  "
      f"MAPE={metrics_prophet['MAPE (%)']:.1f}%  R²={metrics_prophet['R²']:.4f}")

# Graphique avec intervalles de confiance Prophet
prophet_test_fc = fc_prophet.iloc[-HORIZON:]

fig = go.Figure()
fig.add_trace(go.Scatter(x=train['ds'], y=train['y'], mode='lines', name='Train', line=dict(color='steelblue')))
fig.add_trace(go.Scatter(x=test['ds'],  y=test['y'],  mode='lines', name='Réel',  line=dict(color='orange')))
fig.add_trace(go.Scatter(
    x=pd.concat([pd.Series(prophet_test_fc['ds']), pd.Series(prophet_test_fc['ds'].values[::-1])]),
    y=pd.concat([pd.Series(prophet_test_fc['yhat_upper']), pd.Series(prophet_test_fc['yhat_lower'].values[::-1])]),
    fill='toself', fillcolor='rgba(0,128,0,0.12)', line=dict(color='rgba(255,255,255,0)'),
    name='IC 80%'))
fig.add_trace(go.Scatter(x=prophet_test_fc['ds'], y=prophet_test_fc['yhat'], mode='lines',
                         name='Prophet', line=dict(color='green', dash='dash')))
fig.update_layout(title=f"Prophet | R²={metrics_prophet['R²']:.4f} | MAE={metrics_prophet['MAE']:.1f}€",
                  xaxis_title='Date', yaxis_title='Coût (€)', height=380)
fig.show()

---
## Benchmark — Tableau de performance & Ranking

In [ ]:
bench = pd.DataFrame(results)

# Score composite : rang moyen sur MAE, RMSE, MAPE (plus bas = mieux), R² (plus haut = mieux)
bench['Rank_MAE']  = bench['MAE'].rank()
bench['Rank_RMSE'] = bench['RMSE'].rank()
bench['Rank_MAPE'] = bench['MAPE (%)'].rank()
bench['Rank_R2']   = bench['R²'].rank(ascending=False)
bench['Score']     = (bench['Rank_MAE'] + bench['Rank_RMSE'] + bench['Rank_MAPE'] + bench['Rank_R2']) / 4
bench = bench.sort_values('Score').reset_index(drop=True)

display_cols = ['Model', 'MAE', 'RMSE', 'MAPE (%)', 'R²', 'Score']
print('═' * 68)
print('   BENCHMARK — Classement des modèles (Score = rang moyen)')
print('═' * 68)
print(f"  {'#':<3} {'Modèle':<14} {'MAE':>8} {'RMSE':>8} {'MAPE':>8} {'R²':>8} {'Score':>7}")
print('-' * 68)
for i, row in bench.iterrows():
    star = ' ★' if i == 0 else ''
    print(f"  {i+1:<3} {row['Model']:<14} {row['MAE']:>7.2f}€ {row['RMSE']:>7.2f}€ "
          f"{row['MAPE (%)']:>7.1f}% {row['R²']:>8.4f} {row['Score']:>7.2f}{star}")
print('═' * 68)
best_model = bench.iloc[0]['Model']
print(f"\n  Meilleur modèle : {best_model}")

bench[display_cols]

In [ ]:
# ── Graphique métriques comparatives ─────────────────────────────────────────
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=['MAE (€) — moins = mieux', 'RMSE (€) — moins = mieux',
                    'MAPE (%) — moins = mieux', 'R² — plus = mieux']
)

palette = px.colors.qualitative.Bold
models  = bench['Model'].tolist()
colors  = [palette[i % len(palette)] for i in range(len(models))]

metrics_map = [
    ('MAE',      1, 1, False),
    ('RMSE',     1, 2, False),
    ('MAPE (%)', 2, 1, False),
    ('R²',       2, 2, True),
]
for col, row, col_idx, higher_better in metrics_map:
    vals = bench[col].tolist()
    bar_colors = []
    for i, v in enumerate(vals):
        if higher_better:
            bar_colors.append('gold' if i == 0 else colors[i])
        else:
            bar_colors.append('gold' if i == 0 else colors[i])
    fig.add_trace(
        go.Bar(x=models, y=vals, marker_color=bar_colors,
               text=[f'{v:.2f}' for v in vals], textposition='outside'),
        row=row, col=col_idx
    )

fig.update_layout(title='Comparaison des métriques — tous modèles (or = meilleur)',
                  showlegend=False, height=600)
fig.show()

In [ ]:
# ── Graphique comparatif des prévisions vs réel ───────────────────────────────
fig = go.Figure()

# Derniers 30 jours de train pour le contexte visuel
context = train.iloc[-30:]
fig.add_trace(go.Scatter(x=context['ds'], y=context['y'], mode='lines',
                         name='Train (contexte)', line=dict(color='steelblue', width=1.5)))
fig.add_trace(go.Scatter(x=test['ds'], y=test['y'], mode='lines+markers',
                         name='Réel (test)', line=dict(color='black', width=2.5),
                         marker=dict(size=5)))

line_styles = ['dash', 'dot', 'dashdot', 'longdash', 'longdashdot']
model_palette = {'AutoARIMA': 'crimson', 'AutoETS': 'green', 'AutoTheta': 'purple',
                 'TimesNet': 'darkred', 'N-HiTS': 'teal', 'Prophet': 'darkorange'}

for i, (mname, preds) in enumerate(forecasts_all.items()):
    fig.add_trace(go.Scatter(
        x=dates_test, y=preds, mode='lines',
        name=mname,
        line=dict(color=model_palette.get(mname, palette[i]), dash=line_styles[i % len(line_styles)], width=1.8)
    ))

fig.update_layout(
    title='Prévisions de tous les modèles vs Réel (zone test)',
    xaxis_title='Date', yaxis_title='Coût (€)',
    hovermode='x unified', height=500
)
fig.show()

---
## Prévision J+30 — Meilleur modèle

On ré-entraîne le meilleur modèle sur **toutes** les données disponibles (train + test) pour prédire les 30 prochains jours.

In [ ]:
print(f'  Meilleur modèle sélectionné : {best_model}')
full_sf  = daily.rename(columns={'ds': 'ds', 'y': 'y'}).assign(unique_id='total_cost')

last_date  = daily['ds'].max()
future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=HORIZON, freq='D')

if best_model in ('AutoARIMA', 'AutoETS', 'AutoTheta'):
    if best_model == 'AutoARIMA':
        model_final = StatsForecast(models=[AutoARIMA(season_length=7, approximation=True)], freq='D', n_jobs=-1, verbose=False)
    elif best_model == 'AutoETS':
        model_final = StatsForecast(models=[AutoETS(season_length=7)], freq='D', n_jobs=-1, verbose=False)
    else:
        model_final = StatsForecast(models=[AutoTheta(season_length=7)], freq='D', n_jobs=-1, verbose=False)
    model_final.fit(full_sf)
    fc_future = model_final.predict(h=HORIZON)
    y_future = fc_future[best_model].values

elif best_model == 'TimesNet':
    timesnet_final = TimesNet(h=HORIZON, input_size=INPUT_SIZE, encoder_layers=2,
                               d_model=32, d_ff=64, dropout=0.1, top_k=3, num_kernels=4,
                               max_steps=300, batch_size=8, random_seed=42, scaler_type='standard')
    nf_final = NeuralForecast(models=[timesnet_final], freq='D')
    nf_final.fit(full_sf)
    fc_future = nf_final.predict()
    y_future = fc_future['TimesNet'].values

elif best_model == 'N-HiTS':
    nhits_final = NHITS(h=HORIZON, input_size=2*HORIZON, stack_types=['identity','identity','identity'],
                         n_blocks=[1,1,1], mlp_units=[[256,256],[256,256],[256,256]],
                         interpolation_mode='linear', dropout_prob_theta=0.1,
                         max_steps=300, batch_size=8, random_seed=42, scaler_type='standard')
    nf_final = NeuralForecast(models=[nhits_final], freq='D')
    nf_final.fit(full_sf)
    fc_future = nf_final.predict()
    y_future = fc_future['NHITS'].values

elif best_model == 'Prophet':
    m_final = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=False,
                      changepoint_prior_scale=0.1, seasonality_prior_scale=5)
    m_final.fit(daily.rename(columns={'ds': 'ds', 'y': 'y'}))
    future_df = m_final.make_future_dataframe(periods=HORIZON)
    fc_future_prophet = m_final.predict(future_df)
    y_future = fc_future_prophet['yhat'].iloc[-HORIZON:].values

print(f'  Prévision sur {HORIZON} jours : {future_dates[0].date()} → {future_dates[-1].date()}')
print(f'  Total prévu  : {y_future.sum():,.2f} €')
print(f'  Moyenne/jour : {y_future.mean():,.2f} €/j')

fig = go.Figure()
context_all = daily.iloc[-60:]
fig.add_trace(go.Scatter(x=context_all['ds'], y=context_all['y'], mode='lines',
                         name='Historique', line=dict(color='steelblue', width=2)))
fig.add_trace(go.Scatter(x=future_dates, y=y_future, mode='lines+markers',
                         name=f'Prévision {best_model}',
                         line=dict(color='crimson', dash='dash', width=2),
                         marker=dict(size=5)))
fig.add_vrect(x0=future_dates[0], x1=future_dates[-1],
              fillcolor='rgba(255,0,0,0.05)', line_width=0,
              annotation_text='Zone prévision', annotation_position='top left')
fig.update_layout(
    title=f'Prévision J+{HORIZON} — {best_model} | Total estimé : {y_future.sum():,.0f} €',
    xaxis_title='Date', yaxis_title='Coût (€)', height=420
)
fig.show()

---
## Prévision par service — Top 5

On applique le meilleur modèle (ou AutoARIMA par défaut pour sa robustesse) sur les 5 services les plus coûteux.

In [ ]:
service_cols = [c for c in per_service.columns if c != 'ds']
top5_services = (
    per_service[service_cols].sum().sort_values(ascending=False).head(5).index.tolist()
)
print(f'  Top 5 services : {top5_services}')

service_forecasts = {}
n_svc_train = len(per_service) - HORIZON

for svc in top5_services:
    svc_series = per_service[['ds', svc]].rename(columns={svc: 'y'})
    svc_train  = svc_series.iloc[:n_svc_train].assign(unique_id=svc)

    sf_svc = StatsForecast(
        models=[AutoARIMA(season_length=7, approximation=True)],
        freq='D', n_jobs=-1, verbose=False
    )
    sf_svc.fit(svc_train)
    fc_svc = sf_svc.predict(h=HORIZON)
    y_svc_pred = fc_svc['AutoARIMA'].values

    y_svc_true = svc_series.iloc[-HORIZON:]['y'].values
    m = compute_metrics(y_svc_true, y_svc_pred, svc)
    service_forecasts[svc] = {'pred': y_svc_pred, 'true': y_svc_true, 'metrics': m}
    print(f"  {svc[:30]:<32} MAE={m['MAE']:.2f}€  R²={m['R²']:.4f}")

# Chart multi-services
fig = make_subplots(rows=5, cols=1, shared_xaxes=True,
                    subplot_titles=[s[:35] for s in top5_services], vertical_spacing=0.04)
color_list = px.colors.qualitative.Bold
for i, svc in enumerate(top5_services, start=1):
    svc_full = per_service[['ds', svc]].rename(columns={svc: 'y'})
    train_svc_full = svc_full.iloc[:-HORIZON]
    fig.add_trace(go.Scatter(x=train_svc_full['ds'], y=train_svc_full['y'], mode='lines',
                             name='Train', line=dict(color='steelblue', width=1),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=svc_full.iloc[-HORIZON:]['ds'], y=service_forecasts[svc]['true'],
                             mode='lines', name='Réel', line=dict(color='orange', width=1.5),
                             showlegend=(i==1)), row=i, col=1)
    fig.add_trace(go.Scatter(x=svc_full.iloc[-HORIZON:]['ds'], y=service_forecasts[svc]['pred'],
                             mode='lines', name='Prévision', line=dict(color=color_list[i-1], dash='dash'),
                             showlegend=(i==1)), row=i, col=1)

fig.update_layout(title='Prévision AutoARIMA par service — Top 5', height=1000, showlegend=True)
fig.show()

---
## Résumé exécutif

In [ ]:
print('═' * 70)
print('   RÉSUMÉ BENCHMARK — Coûts GCP Veolia')
print('═' * 70)
print(f'  Données       : {len(daily)} jours ({daily["ds"].min().date()} → {daily["ds"].max().date()})')
print(f'  Horizon test  : {HORIZON} jours')
print(f'  Coût réel moy.: {y_test.mean():.2f} €/jour  |  Total test : {y_test.sum():.2f} €')
print()
print('  Classement final :')
for i, row in bench.iterrows():
    medal = ['🥇', '🥈', '🥉', '  4.', '  5.', '  6.'][i]
    print(f"  {medal} {row['Model']:<14}  R²={row['R²']:+.4f}  MAE={row['MAE']:.1f}€  MAPE={row['MAPE (%)']:.1f}%")
print()
print(f'  Meilleur modèle   : {best_model}')
print(f'  Prévision J+{HORIZON} : {y_future.sum():,.0f} € total  |  {y_future.mean():.0f} €/jour en moyenne')
print('═' * 70)